In [ ]:
# This notebook is based on the analysis of 10-fold CV  and the finall SHAP analysis on HANNA dataset. 

In [ ]:
import os
import pandas as pd
import pickle
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from statsmodels.tools import add_constant
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr

# === Inserted seq_features function here (no more external import needed) ===
def seq_features(sequence, single=False, promoter='u6'):
    seq = sequence.upper().replace('U', 'T')
    feats = {}

    feats['GC binary'] = int((seq.count('G') + seq.count('C')) / len(seq) > 0.5)
    feats['AG count'] = seq.count('AG')
    feats['AA count'] = seq.count('AA')
    feats['TTT count'] = seq.count('TTT')
    feats['AAA count'] = seq.count('AAA')
    feats['ATT count'] = seq.count('ATT')
    feats['CTT count'] = seq.count('CTT')

    for pos in [24, 21, 20]:
        feats[f'G_{pos}'] = 1 if len(seq) > pos and seq[pos] == 'G' else 0
    for pos in [20, 19, 18, 3, 1]:
        feats[f'C_{pos}'] = 1 if len(seq) > pos and seq[pos] == 'C' else 0
    for pos in [16, 13]:
        feats[f'A_{pos}'] = 1 if len(seq) > pos and seq[pos] == 'A' else 0

    feats['A_middle'] = seq[10:13].count('A') if len(seq) >= 13 else 0
    feats['G_middle'] = seq[10:13].count('G') if len(seq) >= 13 else 0
    feats['T_end'] = 1 if seq.endswith('T') else 0
    feats['TG_end'] = 1 if seq.endswith('TG') else 0
    feats['AC_end'] = 1 if seq.endswith('AC') else 0
    feats['CA_19'] = 1 if len(seq) > 19 and seq[19:21] == 'CA' else 0
    feats['CT_13'] = 1 if len(seq) > 13 and seq[13:15] == 'CT' else 0
    feats['GT_pos'] = 1 if 'GT' in seq else 0
    feats['CCC_18'] = 1 if len(seq) > 18 and seq[18:21] == 'CCC' else 0

    feats['Motifs'] = sum([seq.count(motif) for motif in ['TTT', 'GGG', 'AAA']])
    feats['Polynucleotides'] = sum([seq.count(nt*4) for nt in 'ATGC'])

    return pd.DataFrame([feats])

# === Your original unchanged workflow ===
def detect_sequence_column(df):
    for col in df.columns:
        if "sequence" in col.lower():
            return col
    raise KeyError("No sequence column found!")

def extract_features(data, sequence_col, promoter='u6'):
    results = []
    for _, row in data.iterrows():
        sequence = row[sequence_col]
        features = seq_features(sequence, single=True, promoter=promoter)
        features.insert(0, 'Sequence', sequence)
        results.append(features)
    return pd.concat(results, ignore_index=True)

def retrain_with_cv(data_path, promoter='u6'):
    print("Loading data...")
    df = pd.read_csv(data_path)
    sequence_col = detect_sequence_column(df)
    print(f"Detected sequence column: {sequence_col}")

    df = df.dropna(subset=['Prediction scores']).reset_index(drop=True)

    print("Extracting features...")
    features_df = extract_features(df, sequence_col, promoter=promoter)

    expected_features = [
        'GC binary', 'AG count', 'AA count', 'TTT count', 'AAA count',
        'ATT count', 'CTT count', 'G_24', 'G_21', 'G_20', 'C_20', 'C_19',
        'C_18', 'C_3', 'C_1', 'A_16', 'A_13', 'A_middle', 'G_middle', 'T_end',
        'TG_end', 'AC_end', 'CA_19', 'CT_13', 'GT_pos', 'CCC_18', 'Motifs',
        'Polynucleotides'
    ]
    X = features_df[expected_features]
    X = add_constant(X, has_constant='add')
    y = df['Prediction scores']

    output_dir = os.path.join(os.path.expanduser("~"), "Downloads", "10_folds_CRISPRedict")
    os.makedirs(output_dir, exist_ok=True)

    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    all_test_dfs = []
    all_y_test = []
    all_y_test_pred = []
    pearson_train_scores = []

    for i, (train_index, test_index) in enumerate(kf.split(X), start=1):
        print(f"Processing Fold {i}...")
        fold_dir = os.path.join(output_dir, f"Fold{i}")
        os.makedirs(fold_dir, exist_ok=True)

        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        model = LinearRegression()
        model.fit(X_train, y_train)

        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        pearson_train = pearsonr(y_train, y_train_pred)[0]
        pearson_train_scores.append(pearson_train)

        with open(os.path.join(fold_dir, f'CRISPRedict_Fold{i}_Model.pkl'), 'wb') as f:
            pickle.dump(model, f)

        train_df = features_df.iloc[train_index].copy()
        train_df['Prediction_GT'] = y_train.values
        train_df['Prediction_Model'] = y_train_pred
        train_df['Fold'] = i
        train_df['Cleaned_Sequence'] = df.iloc[train_index][sequence_col].values
        train_df.to_excel(os.path.join(fold_dir, f"train_fold{i}.xlsx"), index=False)

        test_df = features_df.iloc[test_index].copy()
        test_df['Prediction_GT'] = y_test.values
        test_df['Prediction_Model'] = y_test_pred
        test_df['Fold'] = i
        test_df['Cleaned_Sequence'] = df.iloc[test_index][sequence_col].values
        test_df.to_excel(os.path.join(fold_dir, f"test_fold{i}.xlsx"), index=False)
        test_df.to_excel(os.path.join(fold_dir, f"test_predictions_fold{i}.xlsx"), index=False)

        all_test_dfs.append(test_df)
        all_y_test.extend(y_test)
        all_y_test_pred.extend(y_test_pred)

    combined_test_df = pd.concat(all_test_dfs, ignore_index=True)
    combined_output_path = os.path.join(output_dir, "CRISPRedict_10Fold_AllTestPredictions.xlsx")
    combined_test_df.to_excel(combined_output_path, index=False)

    print("\n Preview of Combined Predictions:")
    print(combined_test_df[['Cleaned_Sequence', 'Prediction_GT', 'Prediction_Model']].head(15))

    r2_combined = r2_score(all_y_test, all_y_test_pred)
    mse_combined = mean_squared_error(all_y_test, all_y_test_pred)
    rmse_combined = np.sqrt(mse_combined)
    pearson_test = pearsonr(all_y_test, all_y_test_pred)[0]
    avg_pearson_train = np.mean(pearson_train_scores)

    print("\n========== SUMMARY METRICS ==========")
    print(f"{'Metric':<30}{'Value'}")
    print(f"{'Average R² (Test)':<30}{r2_combined:.4f}")
    print(f"{'MSE (Test)':<30}{mse_combined:.4f}")
    print(f"{'RMSE (Test)':<30}{rmse_combined:.4f}")
    print(f"{'Pearson (Test)':<30}{pearson_test:.4f}")
    print(f"{'Average Pearson (Train)':<30}{avg_pearson_train:.4f}")

    print(f"\nCombined test predictions saved to:\n  {combined_output_path}")
    print(f"All per-fold results and models saved in:\n  {output_dir}")
    return combined_output_path


# === Your original input path ===
data_path = r'C:\Users\faiza\Downloads\All excel files training testing\Data sets for all tools\30nt crispredict scores data - Copy - Hanna - Copy.csv'

# Run
combined_file = retrain_with_cv(data_path, promoter='u6')


In [ ]:
import os
import pandas as pd 
import pickle
import numpy as np
import shap
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from statsmodels.tools import add_constant
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr

# === REPLACEMENT FOR MISSING MODULE: features.py ===                code for finall shap values 7-13-2025
def seq_features(sequence, single=False, promoter='u6'):
    seq = sequence.upper().replace('U', 'T')
    feats = {}

    feats['GC binary'] = int((seq.count('G') + seq.count('C')) / len(seq) > 0.5)
    feats['AG count'] = seq.count('AG')
    feats['AA count'] = seq.count('AA')
    feats['TTT count'] = seq.count('TTT')
    feats['AAA count'] = seq.count('AAA')
    feats['ATT count'] = seq.count('ATT')
    feats['CTT count'] = seq.count('CTT')
    
    # Position-specific features
    for pos in [24, 21, 20]:
        feats[f'G_{pos}'] = 1 if len(seq) > pos and seq[pos] == 'G' else 0
    for pos in [20, 19, 18, 3, 1]:
        feats[f'C_{pos}'] = 1 if len(seq) > pos and seq[pos] == 'C' else 0
    for pos in [16, 13]:
        feats[f'A_{pos}'] = 1 if len(seq) > pos and seq[pos] == 'A' else 0
    
    feats['A_middle'] = seq[10:13].count('A') if len(seq) >= 13 else 0
    feats['G_middle'] = seq[10:13].count('G') if len(seq) >= 13 else 0
    feats['T_end'] = 1 if seq.endswith('T') else 0
    feats['TG_end'] = 1 if seq.endswith('TG') else 0
    feats['AC_end'] = 1 if seq.endswith('AC') else 0
    feats['CA_19'] = 1 if len(seq) > 19 and seq[19:21] == 'CA' else 0
    feats['CT_13'] = 1 if len(seq) > 13 and seq[13:15] == 'CT' else 0
    feats['GT_pos'] = 1 if 'GT' in seq else 0
    feats['CCC_18'] = 1 if len(seq) > 18 and seq[18:21] == 'CCC' else 0

    feats['Motifs'] = sum([seq.count(motif) for motif in ['TTT', 'GGG', 'AAA']])
    feats['Polynucleotides'] = sum([seq.count(nt*4) for nt in 'ATGC'])

    return pd.DataFrame([feats])


# === Your Original Code (No logic changed) ===
def detect_sequence_column(df):
    for col in df.columns:
        if "sequence" in col.lower():
            return col
    raise KeyError("No sequence column found!")

def extract_features(data, sequence_col, promoter='u6'):
    results = []
    for _, row in data.iterrows():
        sequence = row[sequence_col]
        features = seq_features(sequence, single=True, promoter=promoter)
        features.insert(0, 'Sequence', sequence)
        results.append(features)
    return pd.concat(results, ignore_index=True)

def retrain_with_shap(data_path, promoter='u6'):
    print("Loading data...")
    df = pd.read_csv(data_path)
    sequence_col = detect_sequence_column(df)
    print(f"Detected sequence column: {sequence_col}")

    df = df.dropna(subset=['Prediction scores']).reset_index(drop=True)

    print("Extracting features...")
    features_df = extract_features(df, sequence_col, promoter=promoter)

    expected_features = [
        'GC binary', 'AG count', 'AA count', 'TTT count', 'AAA count',
        'ATT count', 'CTT count', 'G_24', 'G_21', 'G_20', 'C_20', 'C_19',
        'C_18', 'C_3', 'C_1', 'A_16', 'A_13', 'A_middle', 'G_middle', 'T_end',
        'TG_end', 'AC_end', 'CA_19', 'CT_13', 'GT_pos', 'CCC_18', 'Motifs',
        'Polynucleotides'
    ]

    X = features_df[expected_features]
    X = add_constant(X, has_constant='add')
    y = df['Prediction scores']

    # Create output directory
    output_dir = os.path.join(os.path.expanduser("~"), "Downloads", "CRISPRedict_SHAP")
    os.makedirs(output_dir, exist_ok=True)

    print("Training model...")
    model = LinearRegression()
    model.fit(X, y)

    y_pred = model.predict(X)

    r2 = r2_score(y, y_pred)
    mse = mean_squared_error(y, y_pred)
    rmse = np.sqrt(mse)
    pearson_corr = pearsonr(y, y_pred)[0]

    print("\n========== METRICS ==========")
    print(f"R²: {r2:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"Pearson: {pearson_corr:.4f}")

    # Save model
    model_path = os.path.join(output_dir, 'CRISPRedict_Model.pkl')
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)

    print(f"\nModel saved to: {model_path}")

    # SHAP Analysis (only for first sequence)
    print("\nComputing SHAP values for first sequence...")
    explainer = shap.Explainer(model, X)
    shap_values = explainer(X)

    first_idx = 0
    shap.plots.waterfall(shap_values[first_idx], max_display=15, show=False)

    waterfall_path = os.path.join(output_dir, 'SHAP_Waterfall_First_Sequence.png')
    plt.savefig(waterfall_path, bbox_inches='tight', dpi=300)
    plt.show()
    plt.close()

    print(f"SHAP Waterfall plot saved to: {waterfall_path}")

    # Combine all for Excel export
    full_results_df = features_df.copy()
    full_results_df['Prediction_GT'] = y.values
    full_results_df['Prediction_Model'] = y_pred

    for i in range(len(shap_values[0].values)):
        full_results_df[f'SHAP_{X.columns[i]}'] = [val.values[i] for val in shap_values]

    excel_output_path = os.path.join(output_dir, "CRISPRedict_SHAP_Results.xlsx")
    full_results_df.to_excel(excel_output_path, index=False)

    print(f"\nFull results saved to: {excel_output_path}")

    # Print predictions for first five sequences
    print("\n========== First 5 Predictions ==========")
    preview_df = full_results_df[['Sequence', 'Prediction_GT', 'Prediction_Model']].head(5)
    print(preview_df)

    return model_path, waterfall_path, excel_output_path


# === Your original input path ===
data_path = r'C:\Users\faiza\Downloads\All excel files training testing\Data sets for all tools\30nt crispredict scores data - Copy - Hanna - Copy.csv'

# Run
model_file, shap_plot_file, excel_file = retrain_with_shap(data_path, promoter='u6')
